In [1]:
! python -m pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 22.1 MB/s  0:00:00


In [3]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# Lettura file CSV
df_t = pd.read_csv("pc_trasporto.csv", index_col=0)
df_p = pd.read_csv("pc_produzione.csv", index_col=0)

# Definizione insiemi
SI = df_t.index.drop("domanda").to_list()
BJ = df_t.columns.tolist()
A = df_p["capacita"].to_dict()
W = df_p["costo"].to_dict()
B = df_t.loc["domanda"].to_dict()
C = df_t.drop(index="domanda")

print("Stabilimenti:", SI)
print("Banche:", BJ)
print("Capacità produttive:", A)
print("Domanda:", B)
print("Costi di produzione:", W)

# Creazione modello
m = gp.Model("pc")
m.setParam("OutputFlag", 0)

# Variables: x[i,j] = shipped from stabilimento i to banca j
x = m.addVars(SI, BJ, vtype=GRB.INTEGER, lb=0, name="x")

# Definizione vincoli
for i in SI:
    m.addConstr(gp.quicksum(x[i, j] for j in BJ) <= A[i], name=f"capacita_[{i}]")

for j in BJ:
    m.addConstr(gp.quicksum(x[i, j] for i in SI) == B[j], name=f"domanda_[{j}]")

# Definizione O.f.
m.setObjective(
    gp.quicksum((W[i] + C.loc[i, j]) * x[i, j] for i in SI for j in BJ), GRB.MINIMIZE
)

#estensione 1
stabilimento_protetto = 'it'
alpha = 0.25
totProdProtetto = gp.quicksum(x[stabilimento_protetto,j] for j in BJ)
totProd = gp.quicksum(x[i,j] for i in SI for j in BJ)
m.addConstr(totProdProtetto >= alpha * totProd)

#estensione 2
beta = 0.30
for j in BJ:
    if j != stabilimento_protetto:
      m.addConstr(totProdProtetto >= beta * gp.quicksum( x[i,j] for i in SI ) )


# Ottimizzazione
m.optimize()

# Stampa risultati
if m.status == GRB.OPTIMAL:
    print("\n=== RISULTATI ===")
    print(f"Valore funzione obiettivo: {m.objVal:.2f}\n")

    for i in SI:
        for j in BJ:
            if x[i, j].X > 1e-6:  # mostra solo variabili positive
                print(f"Quantità spedita da {i} a {j}: {x[i, j].X:.0f}")

    print("\nCapacità utilizzate:") #punto 4
    for i in SI:
        used = sum(x[i, j].X for j in BJ)
        print(f"{i}: {used:.0f} / {A[i]}")

    print("\nDomanda soddisfatta:")
    for j in BJ:
        got = sum(x[i, j].X for i in SI)
        print(f"{j}: {got:.0f} (domanda {B[j]:.0f})")
else:
    print("Nessuna soluzione ottimale trovata. Stato:", m.status)


#estensione 5
objPrima = m.objVal
stabilimento_potenziato = 'cn'
potenziamento_quanto = 5000
potenziamento_costo = 4000
cpot = m.getConstrByName(f"capacita_[{stabilimento_potenziato}]")
cpot.RHS += potenziamento_quanto
m.update()
m.optimize()
objDopo = m.objVal
print(f"Da {objPrima} a {objDopo}: Delta costo di {objDopo - objPrima} {"" if objDopo < objPrima - 5000 else "non"} giustifica")
cpot.RHS -= potenziamento_quanto
m.update()

#estensione 6
Wbase = W[stabilimento_protetto]
for d in range(0,40,6):
    W[stabilimento_protetto] = Wbase - d
    m.setObjective(
      gp.quicksum((W[i] + C.loc[i, j]) * x[i, j] for i in SI for j in BJ), GRB.MINIMIZE
    )
    m.update()
    m.write(f"pc{d}.lp")
    m.optimize()
    print(f"se il costo in {stabilimento_protetto} è {W[stabilimento_protetto]} , si spende: {m.objVal:.2f}")
    print(f"{sum(x['it',j].X for j in BJ)}: in Italia {
        "più" if sum(x['it',j].X for j in BJ) > sum(x['fr',j].X for j in BJ) else "meno"}")


Stabilimenti: ['it', 'cn', 'fr']
Banche: ['banca1', 'banca2', 'banca3', 'banca4', 'banca5']
Capacità produttive: {'it': 11000, 'cn': 20000, 'fr': 10000}
Domanda: {'banca1': 7100.0, 'banca2': 3400.0, 'banca3': 9700.0, 'banca4': 5200.0, 'banca5': 3050.0}
Costi di produzione: {'it': 220, 'cn': 180, 'fr': 200}

=== RISULTATI ===
Valore funzione obiettivo: 5733615.60

Quantità spedita da it a banca1: 7100
Quantità spedita da it a banca4: 13
Quantità spedita da cn a banca2: 3400
Quantità spedita da cn a banca3: 9700
Quantità spedita da cn a banca4: 3850
Quantità spedita da cn a banca5: 3050
Quantità spedita da fr a banca4: 1337

Capacità utilizzate:
it: 7113 / 11000
cn: 20000 / 20000
fr: 1337 / 10000

Domanda soddisfatta:
banca1: 7100 (domanda 7100)
banca2: 3400 (domanda 3400)
banca3: 9700 (domanda 9700)
banca4: 5200 (domanda 5200)
banca5: 3050 (domanda 3050)
Da 5733615.6 a 5719710.8: Delta costo di -13904.799999999814  giustifica
se il costo in it è 220 , si spende: 5733615.60
7113.0: in It